# VeReMi Final Feature Integrity Audit
Read-only audit before freezing Graph Dataset V1.0.

Input: `GRAPH_FEATURE_ROOT`  
Output: `FEATURE_AUDIT_ROOT`

Run all cells in order. Do not delete or impute anything until warnings are interpreted.


In [ ]:
from pathlib import Path
import sys

# Locate repository root when notebook is executed from notebooks/
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name in {"notebooks", "analysis", "examples"}:
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import duckdb

from src.config import GRAPH_FEATURE_ROOT, FEATURE_AUDIT_ROOT
ROOT=GRAPH_FEATURE_ROOT
ARCH=ROOT/"archives"
OUT=FEATURE_AUDIT_ROOT
OUT.mkdir(parents=True,exist_ok=True)
EXPECTED_ARCHIVES=225
EXPECTED_SNAPS=100
archive_dirs=sorted(p for p in ARCH.iterdir() if p.is_dir())
print("Archives:",len(archive_dirs))
print("Output:",OUT)


In [ ]:
# Completeness, schema, global counts
required=["node_features.parquet","edge_features.parquet","snapshot_features.parquet"]
file_audit=pd.DataFrame([
    {"archive_name":d.name, **{f:(d/f).exists() for f in required}}
    for d in archive_dirs
])
file_audit.to_csv(OUT/"01_file_completeness.csv",index=False)

con=duckdb.connect()
ng=(ARCH/"*"/"node_features.parquet").as_posix()
eg=(ARCH/"*"/"edge_features.parquet").as_posix()
sg=(ARCH/"*"/"snapshot_features.parquet").as_posix()

schemas=[]
for name,g in [("node",ng),("edge",eg),("snapshot",sg)]:
    x=con.execute(f"DESCRIBE SELECT * FROM read_parquet('{g}',union_by_name=true)").df()
    x.insert(0,"table",name)
    schemas.append(x)
schema=pd.concat(schemas,ignore_index=True)
schema.to_csv(OUT/"02_schema_inventory.csv",index=False)

counts=con.execute(f'''
SELECT
 (SELECT COUNT(*) FROM read_parquet('{ng}',union_by_name=true)) node_rows,
 (SELECT COUNT(*) FROM read_parquet('{eg}',union_by_name=true)) edge_rows,
 (SELECT COUNT(*) FROM read_parquet('{sg}',union_by_name=true)) snapshot_rows
''').df()
counts.to_csv(OUT/"03_global_counts.csv",index=False)
display(counts)


In [ ]:
# Snapshot consistency
rows=[]
for d in archive_dirs:
    n=pd.read_parquet(d/"node_features.parquet",columns=["snapshot_id"])
    e=pd.read_parquet(d/"edge_features.parquet",columns=["snapshot_id"])
    s=pd.read_parquet(d/"snapshot_features.parquet",columns=["snapshot_id"])
    rows.append({
        "archive_name":d.name,
        "node_min":n.snapshot_id.min(),"node_max":n.snapshot_id.max(),
        "edge_min":e.snapshot_id.min(),"edge_max":e.snapshot_id.max(),
        "snapshot_min":s.snapshot_id.min(),"snapshot_max":s.snapshot_id.max(),
        "node_snapshot_count":n.snapshot_id.nunique(),
        "edge_snapshot_count":e.snapshot_id.nunique(),
        "snapshot_count":s.snapshot_id.nunique(),
        "has_negative_snapshot":bool((n.snapshot_id<0).any() or (e.snapshot_id<0).any() or (s.snapshot_id<0).any())
    })
snapshot_audit=pd.DataFrame(rows)
snapshot_audit.to_csv(OUT/"04_snapshot_consistency.csv",index=False)
print("Bad snapshot-count archives:",int((snapshot_audit.snapshot_count!=EXPECTED_SNAPS).sum()))
print("Archives with negative snapshots:",int(snapshot_audit.has_negative_snapshot.sum()))
display(snapshot_audit.head())


In [ ]:
# Null and infinity audit
null_records=[]
inf_records=[]
for i,d in enumerate(archive_dirs,1):
    if i==1 or i%25==0: print(f"{i}/{len(archive_dirs)}")
    for table,fn in [("node","node_features.parquet"),("edge","edge_features.parquet"),("snapshot","snapshot_features.parquet")]:
        df=pd.read_parquet(d/fn)
        z=df.isna().sum()
        for col,val in z[z>0].items():
            null_records.append([d.name,table,col,int(val),len(df)])
        num=df.select_dtypes(include=[np.number])
        for col in num.columns:
            a=num[col].to_numpy(dtype=float,copy=False)
            p=int(np.isposinf(a).sum())
            q=int(np.isneginf(a).sum())
            if p or q: inf_records.append([d.name,table,col,p,q])

null_df=pd.DataFrame(null_records,columns=["archive_name","table","column","null_count","row_count"])
inf_df=pd.DataFrame(inf_records,columns=["archive_name","table","column","positive_inf","negative_inf"])
null_df.to_csv(OUT/"05_nulls_by_archive.csv",index=False)
inf_df.to_csv(OUT/"06_infinities.csv",index=False)

if not null_df.empty:
    null_summary=(null_df.groupby(["table","column"],as_index=False)
                  .agg(null_count=("null_count","sum"),affected_archives=("archive_name","nunique"))
                  .sort_values(["table","null_count"],ascending=[True,False]))
else:
    null_summary=pd.DataFrame(columns=["table","column","null_count","affected_archives"])

expected={"delta_speed_magnitude","delta_motion_angle","snapshot_displacement","time_since_previous_pair_message"}
null_summary["classification"]=null_summary["column"].apply(
    lambda x:"EXPECTED_TEMPORAL_INITIALIZATION" if str(x).startswith("prev_") or x in expected else "REQUIRES_REVIEW"
)
null_summary.to_csv(OUT/"07_null_summary.csv",index=False)
display(null_summary)
print("Infinite-value records:",len(inf_df))


In [ ]:
# Physical, causal and graph sanity checks
edge=con.execute(f'''
SELECT
 SUM(CASE WHEN communication_delay<0 THEN 1 ELSE 0 END) negative_delay,
 SUM(CASE WHEN reported_distance_to_receiver<0 THEN 1 ELSE 0 END) negative_distance,
 SUM(CASE WHEN receiver_state_age<0 THEN 1 ELSE 0 END) future_receiver_state,
 SUM(CASE WHEN snapshot_id<0 THEN 1 ELSE 0 END) negative_snapshot
FROM read_parquet('{eg}',union_by_name=true)
''').df().iloc[0]

snap=con.execute(f'''
SELECT
 SUM(CASE WHEN density<0 OR density>1 THEN 1 ELSE 0 END) density_bad,
 SUM(CASE WHEN reciprocity<0 OR reciprocity>1 THEN 1 ELSE 0 END) reciprocity_bad,
 SUM(CASE WHEN average_clustering<0 OR average_clustering>1 THEN 1 ELSE 0 END) clustering_bad,
 SUM(CASE WHEN isolated_node_count<0 OR isolated_node_count>node_count THEN 1 ELSE 0 END) isolated_bad,
 SUM(CASE WHEN largest_component_fraction<0 OR largest_component_fraction>1 THEN 1 ELSE 0 END) lcc_bad
FROM read_parquet('{sg}',union_by_name=true)
''').df().iloc[0]

sanity=pd.DataFrame([
 ["communication_delay >= 0","edge",edge["negative_delay"]],
 ["reported_distance_to_receiver >= 0","edge",edge["negative_distance"]],
 ["receiver state is not from future","edge",edge["future_receiver_state"]],
 ["edge snapshot_id >= 0","edge",edge["negative_snapshot"]],
 ["density in [0,1]","snapshot",snap["density_bad"]],
 ["reciprocity in [0,1]","snapshot",snap["reciprocity_bad"]],
 ["average_clustering in [0,1]","snapshot",snap["clustering_bad"]],
 ["isolated_node_count valid","snapshot",snap["isolated_bad"]],
 ["largest_component_fraction in [0,1]","snapshot",snap["lcc_bad"]],
],columns=["check","table","violations"])
sanity.to_csv(OUT/"08_sanity_checks.csv",index=False)
display(sanity)


In [ ]:
# Duplicate-key audit
dups=[]
for d in archive_dirs:
    n=pd.read_parquet(d/"node_features.parquet",columns=["snapshot_id","vehicle_id"])
    s=pd.read_parquet(d/"snapshot_features.parquet",columns=["snapshot_id"])
    e=pd.read_parquet(d/"edge_features.parquet")
    key=[c for c in ["sender","receiver","messageID","sendTime","rcvTime"] if c in e.columns]
    dups.append({
        "archive_name":d.name,
        "duplicate_node_snapshot_vehicle":int(n.duplicated(["snapshot_id","vehicle_id"]).sum()),
        "duplicate_snapshot_id":int(s.duplicated(["snapshot_id"]).sum()),
        "duplicate_edge_event_key":int(e.duplicated(key).sum()) if key else 0
    })
duplicate_df=pd.DataFrame(dups)
duplicate_df.to_csv(OUT/"09_duplicate_keys.csv",index=False)
display(duplicate_df.drop(columns="archive_name").sum().to_frame("total"))


In [ ]:
# Label distribution
edge_cols=schema.loc[schema["table"]=="edge","column_name"].tolist()
label_candidates=[c for c in edge_cols if any(t in c.lower() for t in ["attack","label","target","malicious","class"])]
print("Potential label columns:",label_candidates)

parts=[]
for col in label_candidates:
    q=con.execute(f'''SELECT CAST("{col}" AS VARCHAR) AS value, COUNT(*) AS count
                      FROM read_parquet('{eg}',union_by_name=true)
                      GROUP BY 1 ORDER BY count DESC''').df()
    q.insert(0,"column",col)
    parts.append(q)
label_dist=pd.concat(parts,ignore_index=True) if parts else pd.DataFrame(columns=["column","value","count"])
label_dist.to_csv(OUT/"10_label_distribution.csv",index=False)
display(label_dist)


In [ ]:
# Feature-role and leakage classification
label_exact={"attackertype","attack","label","target","is_attack","malicious","class"}
ids={"archive_name","vehicle_id","sender","receiver","messageid","snapshot_id"}
times={"time","sendtime","rcvtime","snapshot_start","snapshot_end"}
derived={"labelled_attack_edge_fraction"}

def role(col):
    x=str(col).lower()
    if x in label_exact or "attack_type" in x or "attackertype" in x:
        return "TARGET_LABEL"
    if x in derived:
        return "LEAKAGE_DERIVED_FROM_LABEL"
    if x in ids:
        return "IDENTIFIER_KEY"
    if x in times:
        return "TEMPORAL_METADATA"
    if any(t in x for t in ["label","target","malicious","groundtruth","ground_truth"]):
        return "POTENTIAL_LEAKAGE_REVIEW"
    return "CANDIDATE_MODEL_FEATURE"

roles=schema[["table","column_name"]].drop_duplicates().copy()
roles["role"]=roles["column_name"].map(role)
roles.to_csv(OUT/"11_feature_roles_and_leakage.csv",index=False)

whitelist=roles[roles.role=="CANDIDATE_MODEL_FEATURE"].copy()
excluded=roles[roles.role!="CANDIDATE_MODEL_FEATURE"].copy()
whitelist.to_csv(OUT/"12_candidate_feature_whitelist.csv",index=False)
excluded.to_csv(OUT/"13_excluded_columns.csv",index=False)

print("Candidate features:",len(whitelist))
print("Excluded/non-input columns:",len(excluded))
display(excluded)


In [ ]:
# Final PASS / WARNING / FAIL report
missing=int((~file_audit[required]).sum().sum())
bad_snaps=int((snapshot_audit.snapshot_count!=EXPECTED_SNAPS).sum())
negative=int(snapshot_audit.has_negative_snapshot.sum())
inf_total=0 if inf_df.empty else int(inf_df.positive_inf.sum()+inf_df.negative_inf.sum())
sanity_total=int(sanity.violations.sum())
dup_total=int(duplicate_df.drop(columns="archive_name").sum().sum())
unexpected=0 if null_summary.empty else int(null_summary.loc[null_summary.classification=="REQUIRES_REVIEW","null_count"].sum())

report=[
 ["Archive count","PASS" if len(archive_dirs)==EXPECTED_ARCHIVES else "FAIL",f"{len(archive_dirs)}/{EXPECTED_ARCHIVES}"],
 ["Required files","PASS" if missing==0 else "FAIL",f"{missing} missing"],
 ["100 snapshots/archive","PASS" if bad_snaps==0 else "FAIL",f"{bad_snaps} violations"],
 ["No snapshot -1","PASS" if negative==0 else "FAIL",f"{negative} archives"],
 ["No +/-Inf","PASS" if inf_total==0 else "FAIL",f"{inf_total} values"],
 ["Physical/graph sanity","PASS" if sanity_total==0 else "FAIL",f"{sanity_total} violations"],
 ["Duplicate keys","PASS" if dup_total==0 else "WARNING",f"{dup_total} duplicates"],
 ["Unexpected nulls","PASS" if unexpected==0 else "WARNING",f"{unexpected} cells require review"],
 ["Leakage controls","PASS","Target/leakage-sensitive columns classified for exclusion"]
]
final=pd.DataFrame(report,columns=["check","status","detail"])
overall="FAIL" if (final.status=="FAIL").any() else ("WARNING" if (final.status=="WARNING").any() else "PASS")
final.loc[len(final)]=["OVERALL DATASET INTEGRITY",overall,"Review warnings before freezing Graph Dataset V1.0"]
final.to_csv(OUT/"14_FINAL_AUDIT_REPORT.csv",index=False)

display(final)
print("\nFINAL AUDIT STATUS:",overall)
print("Results saved to:",OUT)


## After the run
Send me the displayed outputs for:
1. null summary,
2. sanity checks,
3. duplicate totals,
4. label distribution,
5. excluded/non-input feature table,
6. final audit report.

A WARNING is not automatically an error; we will interpret it before changing the dataset.
